# 04 - TF-IDF
Term Frequency-Inverse Document Frequency (TF-IDF) feature extraction on M-Tix reviews.

**TF-IDF** measures how important a word is in a document relative to the entire corpus.
- **TF (Term Frequency):** how often a word appears in a document
- **IDF (Inverse Document Frequency):** penalizes words that appear in many documents

In [ ]:
!pip install scikit-learn

In [ ]:
import pandas as pd
import ast
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_csv('mtix_preprocessed.csv')
df['stemmed'] = df['stemmed'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
df['clean_text'] = df['stemmed'].apply(lambda x: ' '.join(x))
print(f'Loaded {len(df)} rows')
df[['content', 'clean_text', 'label']].head()

## 2. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

print('TF-IDF Matrix Shape:', X_tfidf.shape)
print('Sample Features:', tfidf.get_feature_names_out()[:10])

## 3. Top TF-IDF Words

In [ ]:
# Average TF-IDF score per word across all documents
avg_tfidf = X_tfidf.mean(axis=0).A1
feature_names = tfidf.get_feature_names_out()

tfidf_df = pd.DataFrame({'word': feature_names, 'avg_tfidf': avg_tfidf})
tfidf_df = tfidf_df.sort_values('avg_tfidf', ascending=False).head(20)

plt.figure(figsize=(10, 7))
plt.barh(tfidf_df['word'][::-1], tfidf_df['avg_tfidf'][::-1], color='steelblue', edgecolor='black')
plt.title('Top 20 Words by Average TF-IDF Score', fontsize=14)
plt.xlabel('Average TF-IDF Score')
plt.ylabel('Word')
plt.tight_layout()
plt.show()

## 4. TF-IDF by Sentiment

In [ ]:
for label, name in [(1, 'Positive'), (0, 'Negative')]:
 subset = df[df['label'] == label]['clean_text']
 tfidf_sub = TfidfVectorizer(max_features=1000)
 X_sub = tfidf_sub.fit_transform(subset)
 avg = X_sub.mean(axis=0).A1
 feat = tfidf_sub.get_feature_names_out()
 top = pd.DataFrame({'word': feat, 'score': avg}).sort_values('score', ascending=False).head(10)
 print(f'\n--- Top 10 TF-IDF for {name} reviews ---')
 print(top.to_string(index=False))